# 10 — Realocação Inteligente de Pacientes (RQ10)

> **Pergunta:** Como realocar pacientes de litíase renal a centros mais eficientes sem exceder a capacidade hospitalar?
>
> Sub-perguntas: Onde a migração já funciona? Quais hubs têm capacidade ociosa? Qual o impacto potencial?

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
import sys, json
sys.path.insert(0, ".")
from shared import (
    load_kidney, load_hospital_tags, load_cnes_enriched, load_cnes_names,
    hospital_name, city_name, setup_plot_style, save_plot, save_metrics,
    load_metrics, COLORS, DATA_DIR, PLOT_DIR, CATEGORY_MAP, CITY_NAMES,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
cnes = load_cnes_enriched()
names_df = load_cnes_names()

THERAPEUTIC = {"SURGICAL", "INTERVENTIONAL", "SURGICAL_MGMT"}
kidney["is_therapeutic"] = kidney["proc_category"].isin(THERAPEUTIC).astype(int)

# --- Compute efficiency per hospital (same logic as nb05) ---
hp = kidney.copy()
sys_med_cost = hp.groupby("PROC_REA")["VAL_TOT"].transform("median")
sys_med_los = hp.groupby("PROC_REA")["DIAS_PERM"].transform("median")
hp["cost_ratio"] = hp["VAL_TOT"] / sys_med_cost.replace(0, np.nan)
hp["los_ratio"] = hp["DIAS_PERM"] / sys_med_los.replace(0, np.nan)
hp["resolved"] = ((hp["MORTE"] == 0) & (hp["DIAS_PERM"] <= sys_med_los)).astype(int)

hp_agg = hp.groupby(["CNES", "PROC_REA"]).agg(
    n=("VAL_TOT", "count"),
    mean_cost_ratio=("cost_ratio", "mean"),
    mean_los_ratio=("los_ratio", "mean"),
    mortality=("MORTE", "mean"),
    resolution=("resolved", "mean"),
).reset_index()
hp_agg = hp_agg[hp_agg["n"] >= 5]

hosp_eff = hp_agg.groupby("CNES").apply(
    lambda g: pd.Series({
        "n": g["n"].sum(),
        "w_cost_ratio": np.average(g["mean_cost_ratio"], weights=g["n"]),
        "w_los_ratio": np.average(g["mean_los_ratio"], weights=g["n"]),
        "w_mortality": np.average(g["mortality"], weights=g["n"]),
        "w_resolution": np.average(g["resolution"], weights=g["n"]),
    }),
    include_groups=False,
).reset_index()
hosp_eff["efficiency"] = hosp_eff["w_resolution"] / hosp_eff["w_cost_ratio"].replace(0, np.nan)
hosp_eff = hosp_eff.dropna(subset=["efficiency"])
hosp_eff["name"] = hosp_eff["CNES"].apply(lambda c: hospital_name(c, names_df))

# Merge CNES bed data
if "total_beds" in cnes.columns:
    bed_cols = [c for c in ["CNES", "total_beds", "surgical_beds", "icu_beds"] if c in cnes.columns]
    beds = cnes[bed_cols].drop_duplicates("CNES")
    hosp_eff = hosp_eff.merge(beds, on="CNES", how="left")

# City for each hospital
hosp_city = kidney.groupby("CNES")["MUNIC_MOV"].first()
hosp_eff["munic_mov"] = hosp_eff["CNES"].map(hosp_city)
hosp_eff["city"] = hosp_eff["munic_mov"].apply(city_name)

eff_median = hosp_eff["efficiency"].median()

print(f"Loaded {len(kidney):,} admissions, {kidney['CNES'].nunique()} hospitals")
print(f"Efficiency scores for {len(hosp_eff)} hospitals (median = {eff_median:.3f})")

Loaded 206,500 admissions, 510 hospitals
Efficiency scores for 404 hospitals (median = 0.564)


## 1. Matriz Origem-Destino — Para Onde os Pacientes Vão?

In [2]:
# --- Origin-destination flow matrix ---
flow = kidney.groupby(["MUNIC_RES", "MUNIC_MOV"]).agg(
    n=("VAL_TOT", "count"),
    mean_los=("DIAS_PERM", "mean"),
    mortality=("MORTE", "mean"),
    mean_cost=("VAL_TOT", "mean"),
    pct_therapeutic=("is_therapeutic", "mean"),
).reset_index()
flow["origin"] = flow["MUNIC_RES"].apply(city_name)
flow["dest"] = flow["MUNIC_MOV"].apply(city_name)
flow["is_migration"] = flow["MUNIC_RES"] != flow["MUNIC_MOV"]

total_migrated = flow[flow["is_migration"]]["n"].sum()
total_local = flow[~flow["is_migration"]]["n"].sum()
print(f"Total admissions: {len(kidney):,}")
print(f"  Local (same city): {total_local:,} ({total_local/len(kidney)*100:.1f}%)")
print(f"  Migrated: {total_migrated:,} ({total_migrated/len(kidney)*100:.1f}%)")

# Top 20 migration flows
migration_flows = flow[flow["is_migration"]].sort_values("n", ascending=False)
print(f"\n=== TOP 20 FLUXOS DE MIGRAÇÃO ===")
print(f"{'Origem':>25s} -> {'Destino':25s} {'N':>6s} {'LOS':>5s} {'Mort%':>6s} {'Terap%':>6s}")
print("-" * 85)
for _, r in migration_flows.head(20).iterrows():
    print(f"{r['origin'][:25]:>25s} -> {r['dest'][:25]:25s} {r['n']:6,} {r['mean_los']:5.1f} "
          f"{r['mortality']*100:5.2f}% {r['pct_therapeutic']*100:5.1f}%")

# Identify natural hubs
hub_cities = migration_flows.groupby("MUNIC_MOV").agg(
    migrants_received=("n", "sum"),
    origins=("MUNIC_RES", "nunique"),
).reset_index()
hub_cities["city"] = hub_cities["MUNIC_MOV"].apply(city_name)

local_vol = flow[~flow["is_migration"]].groupby("MUNIC_MOV")["n"].sum().rename("local_vol")
hub_cities = hub_cities.merge(local_vol, left_on="MUNIC_MOV", right_index=True, how="left").fillna(0)
hub_cities["total_vol"] = hub_cities["migrants_received"] + hub_cities["local_vol"]
hub_cities["pct_migrant"] = hub_cities["migrants_received"] / hub_cities["total_vol"]
hub_cities = hub_cities.sort_values("migrants_received", ascending=False)

print(f"\n=== TOP 20 HUBS NATURAIS ===")
print(f"{'Cidade':25s} {'Migrantes':>10s} {'Origens':>8s} {'Vol.Total':>10s} {'%Mig':>6s}")
print("-" * 65)
for _, r in hub_cities.head(20).iterrows():
    print(f"{r['city'][:25]:25s} {r['migrants_received']:>10,.0f} {r['origins']:8.0f} "
          f"{r['total_vol']:>10,.0f} {r['pct_migrant']*100:5.1f}%")

Total admissions: 206,500
  Local (same city): 131,207 (63.5%)
  Migrated: 75,293 (36.5%)

=== TOP 20 FLUXOS DE MIGRAÇÃO ===
                   Origem -> Destino                        N   LOS  Mort% Terap%
-------------------------------------------------------------------------------------
                Guarulhos -> São Paulo                  1,424   3.0  0.00%  67.5%
                  Limeira -> Piracicaba                   761   0.9  0.26%  99.5%
                   350410 -> Botucatu                     704   1.5  0.43%  69.5%
          Pindamonhangaba -> Taubaté                      612   2.5  0.00%  97.2%
                   351060 -> Bauru                        592   2.8  0.00%  33.3%
                   353030 -> S.J. Rio Preto               551   2.7  0.54%  85.5%
                   353440 -> São Paulo                    545   2.6  0.37%  62.2%
                   354260 -> 353620                       539   2.1  0.19%  44.5%
                   351060 -> 352250                

In [3]:
# --- Visualization: top 15 hubs ---
top_hubs = hub_cities.head(15).copy()
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

y_pos = range(len(top_hubs))
axes[0].barh(y_pos, top_hubs["local_vol"], color=COLORS["primary"], alpha=0.7, label="Local")
axes[0].barh(y_pos, top_hubs["migrants_received"], left=top_hubs["local_vol"],
             color=COLORS["warning"], alpha=0.7, label="Migrantes")
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(top_hubs["city"].values, fontsize=9)
axes[0].invert_yaxis()
axes[0].set_xlabel("Internacoes")
axes[0].set_title("Volume: Local vs Migrantes")
axes[0].legend(fontsize=9)

axes[1].barh(y_pos, top_hubs["pct_migrant"] * 100, color=COLORS["secondary"], alpha=0.7)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(top_hubs["city"].values, fontsize=9)
axes[1].invert_yaxis()
axes[1].set_xlabel("% Migrantes")
axes[1].set_title("Proporcao de Pacientes Migrantes")
axes[1].axvline(50, color="gray", linestyle="--", alpha=0.5)

fig.suptitle("Hubs Naturais - Cidades que Mais Recebem Pacientes de Fora", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "hub_cities", prefix="10")

  Saved: 10_hub_cities.png


## 2. Capacidade Ociosa — Quem Pode Absorver Mais?

In [4]:
# --- Capacity estimation ---
YEARS = kidney["year"].max() - kidney["year"].min()
if YEARS == 0:
    YEARS = 1
TARGET_OCCUPANCY = 0.80

hosp_vol = kidney.groupby("CNES").agg(
    total_admissions=("VAL_TOT", "count"),
    total_bed_days=("DIAS_PERM", "sum"),
    mean_los=("DIAS_PERM", "mean"),
    pct_therapeutic=("is_therapeutic", "mean"),
).reset_index()
hosp_vol["admissions_per_year"] = hosp_vol["total_admissions"] / YEARS
hosp_vol["bed_days_per_year"] = hosp_vol["total_bed_days"] / YEARS

cap = hosp_eff.merge(hosp_vol[["CNES", "admissions_per_year", "bed_days_per_year",
                                "total_admissions", "pct_therapeutic"]], on="CNES", how="inner")

# Use total_beds (surgical_beds is all zeros in CNES data)
cap["capacity_beds"] = pd.to_numeric(cap.get("total_beds", 0), errors="coerce").fillna(0)

# Kidney-specific bed-day occupancy rate
cap["annual_bed_capacity"] = cap["capacity_beds"] * 365 * TARGET_OCCUPANCY
cap["kidney_bed_day_rate"] = cap["bed_days_per_year"] / cap["annual_bed_capacity"].replace(0, np.nan)
cap["has_capacity_data"] = cap["capacity_beds"] > 0

# Spare capacity for kidney: hospital beds not used by kidney patients
# Since kidney is only one specialty, kidney_bed_day_rate is typically 1-5%
cap["spare_bed_days"] = (cap["annual_bed_capacity"] - cap["bed_days_per_year"]).clip(lower=0)
# Conservative: a hospital can absorb at most 50% more kidney volume than current
cap["spare_kidney_admissions"] = cap["admissions_per_year"] * 0.5
cap_with_beds = cap[cap["has_capacity_data"]].copy()
cap_with_beds["is_efficient"] = cap_with_beds["efficiency"] >= eff_median
cap_with_beds["kidney_occ_pct"] = cap_with_beds["kidney_bed_day_rate"] * 100

print(f"Hospitals with bed data: {len(cap_with_beds)} / {len(cap)}")

efficient_spare = cap_with_beds[cap_with_beds["is_efficient"]].sort_values("kidney_bed_day_rate")

print(f"\n=== EFICIENTES COM MAIOR CAPACIDADE OCIOSA ===")
print(f"{'Hospital':40s} {'Cidade':15s} {'Efic.':>6s} {'Leitos':>6s} {'Vol/ano':>8s} "
      f"{'Ocup.Kid%':>9s} {'Resol%':>7s} {'CustoX':>7s}")
print("-" * 110)
for _, r in efficient_spare.head(20).iterrows():
    print(f"{r['name'][:40]:40s} {r['city'][:15]:15s} {r['efficiency']:6.2f} "
          f"{r['capacity_beds']:6.0f} {r['admissions_per_year']:8.0f} "
          f"{r['kidney_occ_pct']:8.1f}% {r['w_resolution']*100:6.1f}% {r['w_cost_ratio']:6.2f}")

inefficient_high = cap_with_beds[~cap_with_beds["is_efficient"]].sort_values("admissions_per_year", ascending=False)
print(f"\n=== INEFICIENTES COM MAIOR VOLUME (candidatos a exportar) ===")
print(f"{'Hospital':40s} {'Cidade':15s} {'Efic.':>6s} {'Vol/ano':>8s} "
      f"{'Resol%':>7s} {'CustoX':>7s} {'Mort%':>6s}")
print("-" * 100)
for _, r in inefficient_high.head(20).iterrows():
    print(f"{r['name'][:40]:40s} {r['city'][:15]:15s} {r['efficiency']:6.2f} "
          f"{r['admissions_per_year']:8.0f} "
          f"{r['w_resolution']*100:6.1f}% {r['w_cost_ratio']:6.2f} {r['w_mortality']*100:5.2f}%")

Hospitals with bed data: 396 / 404

=== EFICIENTES COM MAIOR CAPACIDADE OCIOSA ===
Hospital                                 Cidade           Efic. Leitos  Vol/ano Ocup.Kid%  Resol%  CustoX
--------------------------------------------------------------------------------------------------------------
QUARTEIRAO DA SAUDE ENGENHEIRO OSVALDO M Cubatão           1.16     12        3      0.0%  100.0%   0.86
AME DR LUIZ ROBERTO BARRADAS BARATA SAO  São Paulo         1.10     24       99      0.0%  100.0%   0.91
AMBULAT DE ESPECIALIDADES E HOSPITAL DIA 352050            0.83     16       11      0.0%  100.0%   1.21
AME AMBULATORIO MED DE ESPECIALIDADES ST 354580            1.12     12        3      0.0%  100.0%   0.90
HC DA FMUSP INSTITUTO DO CORACAO INCOR S São Paulo         0.61    469        1      0.0%   77.8%   1.27
SANTA CASA DE TANABI                     355340            1.48     72        1      0.0%  100.0%   0.68
HOSPITAL INFANTIL CANDIDO FONTOURA SAO P São Paulo         1.05    114

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sc = axes[0].scatter(
    cap_with_beds["kidney_occ_pct"],
    cap_with_beds["efficiency"],
    s=cap_with_beds["admissions_per_year"].clip(upper=500) * 0.5,
    c=cap_with_beds["w_resolution"],
    cmap="RdYlGn", alpha=0.6, edgecolors="gray", linewidth=0.3,
)
axes[0].axhline(eff_median, color="gray", linestyle="--", alpha=0.5,
                label=f"Eficiencia mediana ({eff_median:.2f})")
axes[0].set_xlabel("Ocupacao Kidney (%)")
axes[0].set_ylabel("Score de Eficiencia")
axes[0].set_title("Eficiencia vs Ocupacao (tamanho = volume)")
axes[0].legend(fontsize=9)
plt.colorbar(sc, ax=axes[0], label="Taxa de Resolucao")

top_spare = efficient_spare.head(15).copy()
y = range(len(top_spare))
axes[1].barh(y, top_spare["admissions_per_year"], color=COLORS["success"], alpha=0.7)
axes[1].set_yticks(y)
axes[1].set_yticklabels([f"{r['name'][:30]} ({r['city'][:10]})" for _, r in top_spare.head(15).iterrows()], fontsize=8)
axes[1].invert_yaxis()
axes[1].set_xlabel("Internacoes/Ano (Kidney)")
axes[1].set_title("Top 15 Eficientes com Capacidade Ociosa")

fig.suptitle("Mapa de Capacidade - Quem Pode Absorver Mais Pacientes?", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "capacity_map", prefix="10")

  Saved: 10_capacity_map.png


## 3. Onde a Migração Já Funciona — Evidência Observacional

In [6]:
# --- H10.1: Do patients who migrate to efficient hospitals get better outcomes? ---
kidney_eff = kidney.merge(hosp_eff[["CNES", "efficiency", "w_resolution"]], on="CNES", how="left")
kidney_eff["at_efficient"] = kidney_eff["efficiency"] >= eff_median

groups = {
    "Local @ Eficiente": kidney_eff[(~kidney_eff["migrated"]) & (kidney_eff["at_efficient"])],
    "Local @ Ineficiente": kidney_eff[(~kidney_eff["migrated"]) & (~kidney_eff["at_efficient"])],
    "Migrou -> Eficiente": kidney_eff[(kidney_eff["migrated"]) & (kidney_eff["at_efficient"])],
    "Migrou -> Ineficiente": kidney_eff[(kidney_eff["migrated"]) & (~kidney_eff["at_efficient"])],
}

print("=== OUTCOMES POR GRUPO DE MIGRACAO ===")
print(f"{'Grupo':25s} {'N':>8s} {'LOS':>5s} {'Mort%':>6s} {'Custo':>8s} {'Terap%':>7s} {'D0%':>5s}")
print("-" * 75)
rows = []
for label, g in groups.items():
    n = len(g)
    if n == 0:
        continue
    los = g["DIAS_PERM"].mean()
    mort = g["MORTE"].mean() * 100
    cost = g["VAL_TOT"].mean()
    terap = g["is_therapeutic"].mean() * 100
    d0 = (g["DIAS_PERM"] == 0).mean() * 100
    print(f"{label:25s} {n:>8,} {los:5.1f} {mort:5.2f}% R${cost:>7,.0f} {terap:6.1f}% {d0:4.0f}%")
    rows.append({"group": label, "n": n, "los": los, "mortality": mort/100, "cost": cost, "d0_pct": d0})

g_target = groups["Migrou -> Eficiente"]
g_baseline = groups["Local @ Ineficiente"]
delta_los = delta_mort = delta_cost = 0
if len(g_target) > 30 and len(g_baseline) > 30:
    u_los, p_los = stats.mannwhitneyu(g_target["DIAS_PERM"], g_baseline["DIAS_PERM"], alternative="less")
    u_mort, p_mort = stats.mannwhitneyu(g_target["MORTE"], g_baseline["MORTE"], alternative="less")
    u_cost, p_cost = stats.mannwhitneyu(g_target["VAL_TOT"], g_baseline["VAL_TOT"], alternative="less")

    print(f"\nH10.1: Migrou->Eficiente vs Local@Ineficiente (Mann-Whitney U)")
    print(f"  LOS:         p = {p_los:.2e} {'sig.' if p_los < 0.05 else 'n.s.'}")
    print(f"  Mortalidade: p = {p_mort:.2e} {'sig.' if p_mort < 0.05 else 'n.s.'}")
    print(f"  Custo:       p = {p_cost:.2e} {'sig.' if p_cost < 0.05 else 'n.s.'}")

    delta_los = g_baseline["DIAS_PERM"].mean() - g_target["DIAS_PERM"].mean()
    delta_mort = g_baseline["MORTE"].mean() - g_target["MORTE"].mean()
    delta_cost = g_baseline["VAL_TOT"].mean() - g_target["VAL_TOT"].mean()
    print(f"\n  Delta LOS:  {delta_los:+.1f} dias")
    print(f"  Delta Mort: {delta_mort*100:+.3f} pp")
    print(f"  Delta Custo: R${delta_cost:+,.0f}")

# H10.4: correlation migrant volume vs city efficiency
hub_eff = hub_cities.merge(
    hosp_eff.groupby("munic_mov").agg(
        city_efficiency=("efficiency", "mean"),
        city_resolution=("w_resolution", "mean"),
    ),
    left_on="MUNIC_MOV", right_index=True, how="inner"
)
rho, p = 0, 1
if len(hub_eff) >= 10:
    rho, p = stats.spearmanr(hub_eff["migrants_received"], hub_eff["city_efficiency"])
    print(f"\nH10.4: Migrantes recebidos vs eficiencia da cidade")
    print(f"  Spearman rho = {rho:.3f}, p = {p:.2e}")
    print(f"  -> {'Migrantes ja buscam cidades eficientes' if rho > 0 and p < 0.05 else 'Sem correlacao significativa'}")

=== OUTCOMES POR GRUPO DE MIGRACAO ===
Grupo                            N   LOS  Mort%    Custo  Terap%   D0%
---------------------------------------------------------------------------
Local @ Eficiente           48,593   1.9  0.22% R$    804   58.4%   19%
Local @ Ineficiente         82,614   2.8  0.44% R$    916   56.5%   11%
Migrou -> Eficiente         27,260   2.0  0.20% R$    863   71.9%   13%
Migrou -> Ineficiente       48,033   2.8  0.39% R$  1,032   72.2%    9%

H10.1: Migrou->Eficiente vs Local@Ineficiente (Mann-Whitney U)
  LOS:         p = 1.56e-281 sig.
  Mortalidade: p = 1.13e-08 sig.
  Custo:       p = 1.00e+00 n.s.

  Delta LOS:  +0.7 dias
  Delta Mort: +0.241 pp
  Delta Custo: R$+52

H10.4: Migrantes recebidos vs eficiencia da cidade
  Spearman rho = -0.373, p = 6.15e-08
  -> Sem correlacao significativa


In [7]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

group_data = pd.DataFrame(rows).set_index("group")
order = ["Local @ Ineficiente", "Local @ Eficiente", "Migrou -> Ineficiente", "Migrou -> Eficiente"]
order = [o for o in order if o in group_data.index]
gd = group_data.loc[order]
colors_bar = [COLORS["danger"], COLORS["primary"], COLORS["warning"], COLORS["success"]][:len(order)]

axes[0].barh(range(len(gd)), gd["los"], color=colors_bar, alpha=0.8)
axes[0].set_yticks(range(len(gd)))
axes[0].set_yticklabels(gd.index, fontsize=9)
axes[0].set_xlabel("LOS Medio (dias)")
axes[0].set_title("Tempo de Permanencia")
axes[0].invert_yaxis()

axes[1].barh(range(len(gd)), gd["mortality"] * 100, color=colors_bar, alpha=0.8)
axes[1].set_yticks(range(len(gd)))
axes[1].set_yticklabels(gd.index, fontsize=9)
axes[1].set_xlabel("Mortalidade (%)")
axes[1].set_title("Mortalidade")
axes[1].invert_yaxis()

axes[2].barh(range(len(gd)), gd["cost"], color=colors_bar, alpha=0.8)
axes[2].set_yticks(range(len(gd)))
axes[2].set_yticklabels(gd.index, fontsize=9)
axes[2].set_xlabel("Custo Medio (R$)")
axes[2].set_title("Custo por Internacao")
axes[2].invert_yaxis()

fig.suptitle("Evidencia: Migrar para Hospitais Eficientes Melhora Resultados", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "migration_outcomes", prefix="10")

  Saved: 10_migration_outcomes.png


## 4. Simulacao de Realocacao — Cenarios com Restricao de Capacidade

In [8]:
# --- Reallocation simulation ---
kidney_eff["microregion_res"] = kidney_eff["MUNIC_RES"].str[:4]
kidney_eff["microregion_mov"] = kidney_eff["MUNIC_MOV"].str[:4]

dest_stats = hosp_eff[hosp_eff["efficiency"] >= eff_median].copy()
dest_stats["microregion"] = dest_stats["munic_mov"].str[:4]

dest_stats = dest_stats.merge(hosp_vol[["CNES", "admissions_per_year", "bed_days_per_year"]],
                               on="CNES", how="left", suffixes=("", "_vol"))

# Capacity model: kidney stones use ~2-5% of total bed capacity.
# A hospital can realistically double its kidney caseload without major infrastructure change.
# Constraint: spare = current kidney admissions/year (i.e., can absorb up to 100% more)
dest_stats["spare_admissions"] = dest_stats["admissions_per_year"].clip(lower=0)
dest_stats["has_spare"] = dest_stats["spare_admissions"] > 5

print(f"Efficient hospitals: {len(dest_stats)}")
print(f"  With spare capacity: {dest_stats['has_spare'].sum()}")
print(f"  Total spare admissions (50% safety): {dest_stats['spare_admissions'].sum():,.0f}")

eligible = kidney_eff[
    (~kidney_eff["at_efficient"]) &
    (kidney_eff["is_therapeutic"] == 1)
].copy()
print(f"\nEligible patients (therapeutic @ inefficient): {len(eligible):,}")

# Map microregion -> best destination
micros = eligible["microregion_res"].unique()
micro_dest = {}
for m in micros:
    candidates = dest_stats[(dest_stats["microregion"] == m) & (dest_stats["has_spare"])]
    if len(candidates) > 0:
        micro_dest[m] = candidates.sort_values("efficiency", ascending=False).iloc[0]["CNES"]

print(f"Microregions with eligible patients: {len(micros)}")
print(f"Microregions with available destination: {len(micro_dest)} ({len(micro_dest)/len(micros)*100:.0f}%)")

eligible["dest_cnes"] = eligible["microregion_res"].map(micro_dest)
eligible_matched = eligible.dropna(subset=["dest_cnes"]).copy()
print(f"Patients matched to a destination: {len(eligible_matched):,} ({len(eligible_matched)/len(eligible)*100:.1f}%)")

# Destination outcomes
dest_outcomes = hosp_eff.set_index("CNES")[["w_resolution", "w_cost_ratio", "w_los_ratio", "w_mortality", "efficiency"]].to_dict("index")
eligible_matched["dest_los_ratio"] = eligible_matched["dest_cnes"].map(lambda c: dest_outcomes.get(c, {}).get("w_los_ratio", np.nan))
eligible_matched["dest_cost_ratio"] = eligible_matched["dest_cnes"].map(lambda c: dest_outcomes.get(c, {}).get("w_cost_ratio", np.nan))
eligible_matched["dest_mortality"] = eligible_matched["dest_cnes"].map(lambda c: dest_outcomes.get(c, {}).get("w_mortality", np.nan))

eligible_matched["current_los_ratio"] = eligible_matched["CNES"].map(hosp_eff.set_index("CNES")["w_los_ratio"].to_dict())
eligible_matched["expected_los_factor"] = eligible_matched["dest_los_ratio"] / eligible_matched["current_los_ratio"].replace(0, np.nan)
eligible_matched["expected_los"] = (eligible_matched["DIAS_PERM"] * eligible_matched["expected_los_factor"]).clip(lower=0)
eligible_matched["saved_los"] = eligible_matched["DIAS_PERM"] - eligible_matched["expected_los"]

eligible_matched["current_cost_ratio"] = eligible_matched["CNES"].map(hosp_eff.set_index("CNES")["w_cost_ratio"].to_dict())
eligible_matched["expected_cost_factor"] = eligible_matched["dest_cost_ratio"] / eligible_matched["current_cost_ratio"].replace(0, np.nan)
eligible_matched["expected_cost"] = (eligible_matched["VAL_TOT"] * eligible_matched["expected_cost_factor"]).clip(lower=0)
eligible_matched["saved_cost"] = eligible_matched["VAL_TOT"] - eligible_matched["expected_cost"]

eligible_matched["current_mort_rate"] = eligible_matched["CNES"].map(hosp_eff.set_index("CNES")["w_mortality"].to_dict())
eligible_matched["deaths_avoided"] = eligible_matched["current_mort_rate"] - eligible_matched["dest_mortality"]

Efficient hospitals: 202
  With spare capacity: 104
  Total spare admissions (50% safety): 7,585

Eligible patients (therapeutic @ inefficient): 81,389
Microregions with eligible patients: 155
Microregions with available destination: 49 (32%)
Patients matched to a destination: 72,619 (89.2%)


In [9]:
def run_scenario(matched_df, dest_stats_df, label, max_hubs=None):
    df = matched_df.copy()
    if max_hubs is not None:
        # Select top N destinations that actually appear in matched data, ranked by efficiency
        used_dests = df["dest_cnes"].unique()
        available = dest_stats_df[dest_stats_df["CNES"].isin(used_dests)].sort_values("efficiency", ascending=False)
        top_dest = available.head(max_hubs)
        df = df[df["dest_cnes"].isin(top_dest["CNES"])]

    dest_spare = dest_stats_df.set_index("CNES")["spare_admissions"].to_dict()
    dest_counts = {}
    keep = []
    for idx, row in df.iterrows():
        dest = row["dest_cnes"]
        spare = dest_spare.get(dest, 0)
        used = dest_counts.get(dest, 0)
        if used < spare:
            keep.append(idx)
            dest_counts[dest] = used + 1
    df = df.loc[keep]

    return {
        "label": label,
        "patients": len(df),
        "destinations": df["dest_cnes"].nunique() if len(df) > 0 else 0,
        "bed_days_saved": df["saved_los"].sum() if len(df) > 0 else 0,
        "cost_saved": df["saved_cost"].sum() if len(df) > 0 else 0,
        "deaths_avoided": df["deaths_avoided"].sum() if len(df) > 0 else 0,
        "pct_of_total": len(df) / len(kidney) * 100,
        "pct_bed_days": df["saved_los"].sum() / kidney["DIAS_PERM"].sum() * 100 if len(df) > 0 else 0,
    }

scenarios = [
    run_scenario(eligible_matched, dest_stats, "Conservador (top 10 hubs)", max_hubs=10),
    run_scenario(eligible_matched, dest_stats, "Moderado (top 30 hubs)", max_hubs=30),
    run_scenario(eligible_matched, dest_stats, "Agressivo (todos c/ capacidade)", max_hubs=None),
]

print("=== CENARIOS DE REALOCACAO ===")
print(f"{'Cenario':40s} {'Pacientes':>10s} {'Destinos':>9s} {'Leitos-dia':>12s} {'%LeitTotal':>10s} "
      f"{'Custo Salvo':>12s} {'Mortes Evit.':>12s}")
print("-" * 115)
for s in scenarios:
    print(f"{s['label']:40s} {s['patients']:>10,} {s['destinations']:>9} "
          f"{s['bed_days_saved']:>12,.0f} {s['pct_bed_days']:>9.1f}% "
          f"R${s['cost_saved']:>11,.0f} {s['deaths_avoided']:>12.1f}")

total_bed_days = kidney["DIAS_PERM"].sum()
total_cost = kidney["VAL_TOT"].sum()
total_deaths = kidney["MORTE"].sum()
print(f"\nContexto do sistema:")
print(f"  Total leitos-dia: {total_bed_days:,.0f}")
print(f"  Total custo: R${total_cost:,.0f}")
print(f"  Total obitos: {total_deaths:,}")

=== CENARIOS DE REALOCACAO ===
Cenario                                   Pacientes  Destinos   Leitos-dia %LeitTotal  Custo Salvo Mortes Evit.
-------------------------------------------------------------------------------------------------------------------
Conservador (top 10 hubs)                       294        10           49       0.0% R$    130,042          1.0
Moderado (top 30 hubs)                          564        30           64       0.0% R$    216,337          1.4
Agressivo (todos c/ capacidade)               1,601        49          206       0.0% R$    409,618          2.2

Contexto do sistema:
  Total leitos-dia: 507,465
  Total custo: R$187,823,156
  Total obitos: 714


In [10]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
labels = [s["label"].split("(")[0].strip() for s in scenarios]
colors_sc = [COLORS["success"], COLORS["primary"], COLORS["warning"]]

vals_p = [s["patients"] for s in scenarios]
axes[0].bar(labels, vals_p, color=colors_sc, alpha=0.8)
axes[0].set_ylabel("Pacientes Realocados")
axes[0].set_title("Volume")
for i, s in enumerate(scenarios):
    axes[0].text(i, s["patients"] * 1.03, f"{s['pct_of_total']:.1f}%", ha="center", fontsize=9)
axes[0].set_ylim(0, max(vals_p) * 1.2)

vals_b = [s["bed_days_saved"] for s in scenarios]
axes[1].bar(labels, vals_b, color=colors_sc, alpha=0.8)
axes[1].set_ylabel("Leitos-dia Salvos")
axes[1].set_title("Leitos-dia")
for i, s in enumerate(scenarios):
    axes[1].text(i, s["bed_days_saved"] * 1.03, f"{s['pct_bed_days']:.1f}%", ha="center", fontsize=9)
axes[1].set_ylim(0, max(vals_b) * 1.2)

vals_c = [s["cost_saved"] / 1e6 for s in scenarios]
axes[2].bar(labels, vals_c, color=colors_sc, alpha=0.8)
axes[2].set_ylabel("Custo Economizado (R$ milhões)")
axes[2].set_title("Economia")
for i, s in enumerate(scenarios):
    axes[2].text(i, s["cost_saved"]/1e6 * 1.03, f"R${s['cost_saved']/1e6:.1f}M", ha="center", fontsize=9)
axes[2].set_ylim(0, max(vals_c) * 1.2)

fig.suptitle("Cenários de Realocação — Impacto Estimado", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "reallocation_scenarios", prefix="10")

  Saved: 10_reallocation_scenarios.png


## 5. Hubs Prioritarios — Onde Investir

In [11]:
hub_priority = dest_stats[dest_stats["has_spare"]].merge(
    hub_cities[["MUNIC_MOV", "migrants_received", "origins", "pct_migrant"]],
    left_on="munic_mov", right_on="MUNIC_MOV", how="left"
).fillna(0)

q95_spare = hub_priority["spare_admissions"].quantile(0.95)
q95_mig = max(hub_priority["migrants_received"].quantile(0.95), 1)
hub_priority["priority_score"] = (
    hub_priority["efficiency"] / hub_priority["efficiency"].max() * 0.3 +
    hub_priority["spare_admissions"].clip(upper=q95_spare) / q95_spare * 0.3 +
    hub_priority["migrants_received"].clip(upper=q95_mig) / q95_mig * 0.2 +
    hub_priority["w_resolution"] * 0.2
)
hub_priority = hub_priority.sort_values("priority_score", ascending=False)

print("=== TOP 20 HUBS PRIORITARIOS ===")
print(f"{'Hospital':40s} {'Cidade':15s} {'Efic.':>6s} {'FolgaAdm':>9s} {'MigReceb':>9s} "
      f"{'Resol%':>7s} {'Priority':>8s}")
print("-" * 105)
for _, r in hub_priority.head(20).iterrows():
    print(f"{r['name'][:40]:40s} {r['city'][:15]:15s} {r['efficiency']:6.2f} "
          f"{r['spare_admissions']:9.0f} {r['migrants_received']:9.0f} "
          f"{r['w_resolution']*100:6.1f}% {r['priority_score']:8.3f}")

city_hubs = hub_priority.groupby("city").agg(
    n_hospitals=("CNES", "count"),
    total_spare=("spare_admissions", "sum"),
    mean_efficiency=("efficiency", "mean"),
    total_migrants=("migrants_received", "sum"),
    max_priority=("priority_score", "max"),
).sort_values("max_priority", ascending=False)

print(f"\n=== TOP 15 CIDADES-HUB ===")
print(f"{'Cidade':25s} {'Hospitais':>10s} {'FolgaTotal':>11s} {'Efic.Media':>11s} {'Migrantes':>10s}")
print("-" * 75)
for city_name_val, r in city_hubs.head(15).iterrows():
    print(f"{str(city_name_val)[:25]:25s} {r['n_hospitals']:10.0f} {r['total_spare']:11.0f} "
          f"{r['mean_efficiency']:11.2f} {r['total_migrants']:10.0f}")

=== TOP 20 HUBS PRIORITARIOS ===
Hospital                                 Cidade           Efic.  FolgaAdm  MigReceb  Resol% Priority
---------------------------------------------------------------------------------------------------------
HOSPITAL DIA SANTO AMARO                 São Paulo         0.61       554      9400   90.2%    0.806
HOSP MUN M BOI MIRIM                     São Paulo         0.62       326      9400   68.4%    0.764
AME DR LUIZ ROBERTO BARRADAS BARATA SAO  São Paulo         1.10        99      9400  100.0%    0.728
UNIDADE DE GESTAO ASSISTENCIAL II HOSPIT São Paulo         0.61       222      9400   61.1%    0.678
HOSPITAL DE BASE DE BAURU                Bebedouro         0.71       346      2248   85.4%    0.664
HOSPITAL REGIONAL DO VALE DO PARAIBA     Taubaté           0.63       453      2778   73.3%    0.636
HOSPITAL DR LEOPOLDO BEVILACQUA          353620            0.61       283      2380   79.4%    0.628
HOSPITAL DIA SAO MIGUEL DR TITO LOPES DA São Paulo   

In [12]:
fig, ax = plt.subplots(figsize=(14, 8))
top20 = hub_priority.head(20).copy()
y = range(len(top20))
med_eff = top20["efficiency"].median()
ax.barh(y, top20["priority_score"], color=[
    COLORS["success"] if r["efficiency"] > med_eff else COLORS["primary"]
    for _, r in top20.iterrows()
], alpha=0.8)
ax.set_yticks(y)
ax.set_yticklabels([f"{r['name'][:35]} ({r['city'][:12]})" for _, r in top20.iterrows()], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel("Score de Prioridade")
ax.set_title("Top 20 Hubs Prioritarios para Realocacao", fontsize=14, fontweight="bold")

for i, (_, r) in enumerate(top20.iterrows()):
    ax.text(r["priority_score"] + 0.01, i, f"folga: {r['spare_admissions']:.0f}/ano",
            va="center", fontsize=7, color="gray")

fig.tight_layout()
save_plot(fig, "priority_hubs", prefix="10")

  Saved: 10_priority_hubs.png


## Summary Metrics

In [13]:
metrics = {
    "total_admissions": len(kidney),
    "migration_rate": float(kidney["migrated"].mean()),
    "total_migrated": int(kidney["migrated"].sum()),
    "n_origin_cities": int(kidney["MUNIC_RES"].nunique()),
    "n_dest_cities": int(kidney["MUNIC_MOV"].nunique()),
    "n_migration_flows": int(len(migration_flows)),
    "top_hub_city": str(hub_cities.iloc[0]["city"]) if len(hub_cities) > 0 else "N/A",
    "top_hub_migrants": int(hub_cities.iloc[0]["migrants_received"]) if len(hub_cities) > 0 else 0,
    "efficient_hospitals": int(len(dest_stats)),
    "efficient_with_spare": int(dest_stats["has_spare"].sum()),
    "total_spare_admissions": float(dest_stats["spare_admissions"].sum()),
    "eligible_for_reallocation": len(eligible),
    "matched_to_destination": len(eligible_matched),
    "scenarios": scenarios,
    "h10_1_los_delta": float(delta_los),
    "h10_1_mort_delta": float(delta_mort),
    "h10_4_rho": float(rho),
    "h10_4_p": float(p),
}
save_metrics(metrics, "10_patient_reallocation")
print("Done.")

  Saved metrics: 10_patient_reallocation.json
Done.
